# Satellite Field Boundary Segmentation — Demo

This notebook demonstrates inference on a sample satellite tile and visualizes the results.

### Prerequisites
- Run `uv sync --all-extras` to install dependencies
- Run `bash data/download.sh` to get sample data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import torch

from src.model import build_model
from src.postprocess import threshold_maps

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Load model
model = build_model(in_channels=4)
model = model.to(device)
model.eval()

# TODO: Load checkpoint
# checkpoint = torch.load("checkpoints/best.pt", map_location=device, weights_only=True)
# model.load_state_dict(checkpoint["model_state_dict"])
print("Model loaded (untrained — for demo visualization only)")

In [ ]:
# Load sample tile
input_path = "data/sample_tile.tif"

with rasterio.open(input_path) as ds:
    image = ds.read().astype(np.float32) / 255.0
    transform = ds.transform

print(f"Tile shape: {image.shape} (C, H, W)")

# Display RGB composite
rgb = image[[2, 1, 0], :, :].transpose(1, 2, 0)  # BGR → RGB
rgb = np.clip(rgb, 0, 1)

plt.figure(figsize=(8, 8))
plt.imshow(rgb)
plt.title("Sentinel-2 tile (RGB)")
plt.axis("off")
plt.show()

In [ ]:
# Run inference
tensor = torch.from_numpy(image).unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(tensor)

extent = outputs["extent"].squeeze().cpu().numpy()
boundary = outputs["boundary"].squeeze().cpu().numpy()
distance = outputs["distance"].squeeze().cpu().numpy()

print(f"Extent range: [{extent.min():.3f}, {extent.max():.3f}]")
print(f"Boundary range: [{boundary.min():.3f}, {boundary.max():.3f}]")

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(rgb)
axes[0].set_title("Input (RGB)")
axes[0].axis("off")

im1 = axes[1].imshow(extent, cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("Extent Map")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(boundary, cmap="Reds", vmin=0, vmax=1)
axes[2].set_title("Boundary Map")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2])

im3 = axes[3].imshow(distance, cmap="plasma")
axes[3].set_title("Distance Transform")
axes[3].axis("off")
plt.colorbar(im3, ax=axes[3])

plt.tight_layout()
plt.show()

In [ ]:
# Export to GeoPackage
from src.postprocess import raster_to_geopackage
import geopandas as gpd

output_path = "output/demo_fields.gpkg"
gdf = raster_to_geopackage(extent, boundary, output_path, transform=transform)

print(f"\nExported {len(gdf)} polygons to {output_path}")

# Overlay on satellite image
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb)
if not gdf.empty:
    gdf.plot(ax=ax, edgecolor="red", facecolor="none", linewidth=1.5)
ax.set_title("Predicted Field Boundaries")
ax.axis("off")
plt.show()